# FeedbackIQ — Customer Feedback RAG Chatbot Prototype
**Phase 5: Retrieval-Augmented Generation (RAG) Data & Reasoning Layer**

This notebook demonstrates the end-to-end Python RAG pipeline for customer feedback intelligence.

### Pipeline Architecture:
```text
Product ID
    ↓
Retrieve reviews belonging ONLY to this product (Strict Isolation)
    ↓
Prioritize negative reviews when the question is about complaints
    ↓
Generate embeddings using Sentence Transformers (all-MiniLM-L6-v2)
    ↓
Store & Retrieve embeddings using FAISS (IndexFlatIP)
    ↓
Retrieve top relevant reviews
    ↓
Include sentiment + domain aspect information
    ↓
Send grounded context to an LLM (Modular Layer)
    ↓
Generate structured, verifiable answer with citations
```

---
### Key Architectural Guarantees:
1. **Strict Product Isolation**: The FAISS index is partitioned per `parent_asin`, guaranteeing zero cross-product review leakage.
2. **Intent-Based Sentiment Prioritization**: Questions regarding complaints prioritize negative reviews; questions regarding praises prioritize positive reviews.
3. **Aspect Enrichment**: Every retrieved review is enriched with domain-specific taxonomies (*Electronics* vs. *Fashion*).
4. **Zero-Hallucination Grounded Generation**: Answers cite exact review counts, ratings, aspects, and verbatim customer quotes.

In [1]:
import os
import sys
from pathlib import Path

# Ensure Windows OpenMP DLL compatibility
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Add project root to sys.path
root_path = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(root_path) not in sys.path:
    sys.path.insert(0, str(root_path))

import pandas as pd
from IPython.display import display, Markdown, HTML

from src.retrieval import ReviewRetriever, ASPECT_DISPLAY_NAMES
from src.rag import FeedbackRAGPipeline, GroundedSynthesizerLLM, get_llm

print("FeedbackIQ RAG Environment initialized successfully.")

FeedbackIQ RAG Environment initialized successfully.


## 1. Product Catalog Overview
Inspect the products available in the processed feedback database across **Electronics** and **Fashion**.

In [2]:
retriever = ReviewRetriever()
products = retriever.get_available_products()

catalog_data = []
for p in products:
    catalog_data.append({
        "Product ID (ASIN)": p["parent_asin"],
        "Domain": p["domain"],
        "Product Title": p["title"][:50] + ("..." if len(p["title"]) > 50 else ""),
        "Reviews": p["total_reviews"],
        "Avg Rating": f"{p['average_rating']} ★",
        "Positive %": f"{p['sentiment_percentages']['positive']}%",
        "Neutral %": f"{p['sentiment_percentages']['neutral']}%",
        "Negative %": f"{p['sentiment_percentages']['negative']}%",
    })

df_catalog = pd.DataFrame(catalog_data)
display(HTML("<h3>Available Products</h3>"))
display(df_catalog)

Available Products

Product ID (ASIN),Domain,Product Title,Reviews,Avg Rating,Positive %,Neutral %,Negative %
B07TVHSDMQ,Fashion,Funny Civil Engineers TShirt I'm A Crazy Civil Eng...,246,4.6 ★,88.2%,5.3%,6.5%
B075X8471B,Electronics,"Fire TV Stick with Alexa Voice Remote, streaming m...",144,4.36 ★,82.6%,5.6%,11.8%
B010BWYDYA,Electronics,"Fire Tablet with Alexa, 7"" Display, 16 GB, Blue - ...",91,4.26 ★,80.2%,8.8%,11.0%
B07S764D9V,Electronics,"Panasonic ErgoFit Wired Earbuds, In-Ear Headphones...",78,4.17 ★,78.2%,7.7%,14.1%
B08F6GPRH6,Electronics,"Blink Whole Home Bundle | Video Doorbell System, O...",29,3.55 ★,62.1%,10.3%,27.6%
B07F4P3JH7,Electronics,"Fire HD 8 Tablet (8"" HD Display, 16 GB) - Black (P...",28,4.39 ★,85.7%,3.6%,10.7%
B08YFJYD5Z,Electronics,"All-new Fire HD 10 tablet, 10.1"", 1080p Full HD, 3...",24,4.5 ★,87.5%,8.3%,4.2%
B07PGL2N7J,Electronics,Echo Dot (3rd Gen) - Smart speaker with Alexa - Sa...,23,4.48 ★,87.0%,0.0%,13.0%
B0BMQJYLQV,Electronics,"Cat 6 Ethernet Cable 1 Ft (6Pack), Outdoor&Indoor,...",23,4.7 ★,91.3%,4.3%,4.3%
B0956HNH2Y,Fashion,Fruit of the Loom Women's Eversoft Cotton Brief Un...,20,3.8 ★,65.0%,15.0%,20.0%


## 2. Complaint Query — Electronics (Fire TV Stick)
* **Product**: `B075X8471B` (Fire TV Stick with Alexa Voice Remote)
* **Question**: `"Why are customers unhappy with this product?"`

The system detects complaint intent, prioritizes negative reviews, performs FAISS semantic search, and generates grounded synthesis.

In [3]:
pipeline = FeedbackRAGPipeline(retriever=retriever)

asin_firetv = "B075X8471B"
query_1 = "Why are customers unhappy with this product?"

res_1 = pipeline.answer_question(asin_firetv, query_1, top_k=4)

display(HTML(f"<b>Query Intent Detected:</b> <code>{res_1['retrieval_context']['intent']}</code>"))

# Show retrieved reviews in a table
retrieved_rows = []
for r in res_1["retrieval_context"]["retrieved_reviews"]:
    retrieved_rows.append({
        "Rating": f"{r['rating']} ★",
        "Sentiment": r["sentiment"],
        "Aspects": ", ".join(r["aspect_labels"]),
        "Title": r["title"][:40],
        "Review Snippet": r["text"][:110] + "...",
        "Sim Score": r["similarity_score"],
    })
display(HTML("<h4>Retrieved Isolated Reviews:</h4>"))
display(pd.DataFrame(retrieved_rows))

display(HTML("<h4>Grounded RAG Response:</h4>"))
display(Markdown(res_1["answer"]))

Query Intent Detected: complaints

Retrieved Isolated Reviews:

Rating,Sentiment,Aspects,Title,Review Snippet,Sim Score
2.0 ★,negative,"Customer Service & Returns, Compatibility & Connection",I am returning mine to the store,It pushes their own pay content even if you do not have Amazon Prime. As in there is Prime stuff on the main s...,0.3374
1.0 ★,negative,Durability & Build Quality,Your being ripped off,Great concept but once you stop all the subscriptions your stick doesn't work anymore....,0.3180
2.0 ★,negative,Performance & Functionality,Sporadic and unreliable.,Works when it feels like it....,0.2275
2.0 ★,negative,Compatibility & Connection,Amazing tv,Works great but this is the 2nd one I purchased in 3years. Very upset. Should last longer....,0.2240


Grounded RAG Response:

Based on 4 relevant reviews for "Fire TV Stick with Alexa Voice Remote, streaming media player - Previous Generation" (ASIN: B075X8471B):

### 1. Sentiment Breakdown of Retrieved Reviews
- Negative: 4 (100%) | Neutral: 0 (0%) | Positive: 0 (0%)
- Product Catalog Rating: 4.36 / 5.0 (across 144 total reviews)

### 2. Relevant Aspects Identified
- **Compatibility & Connection**: cited in 2 review(s) (50.0%)
- **Customer Service & Returns**: cited in 1 review(s) (25.0%)
- **Durability & Build Quality**: cited in 1 review(s) (25.0%)
- **Performance & Functionality**: cited in 1 review(s) (25.0%)

### 3. Main Customer Complaints
1. **Customer Service & Returns, Compatibility & Connection**: I am returning mine to the store
2. **Durability & Build Quality**: Your being ripped off
3. **Performance & Functionality**: Sporadic and unreliable.
4. **Compatibility & Connection**: Amazing tv

### 4. Representative Evidence from Customer Reviews
1. *"It pushes their own pay content even if you do not have Amazon Prime. As in there is Prime stuff on the main screen even when you do not have Prime which is odd. It is geared toward Prime people. I am probably going t..."*  
   — **Rating**: 2.0 / 5.0 | **Sentiment**: negative | **Aspects**: [Customer Service & Returns, Compatibility & Connection]
2. *"Great concept but once you stop all the subscriptions your stick doesn't work anymore."*  
   — **Rating**: 1.0 / 5.0 | **Sentiment**: negative | **Aspects**: [Durability & Build Quality]
3. *"Works when it feels like it."*  
   — **Rating**: 2.0 / 5.0 | **Sentiment**: negative | **Aspects**: [Performance & Functionality]
4. *"Works great but this is the 2nd one I purchased in 3years. Very upset. Should last longer."*  
   — **Rating**: 2.0 / 5.0 | **Sentiment**: negative | **Aspects**: [Compatibility & Connection]

## 3. Complaint Query — Panasonic Earbuds
* **Product**: `B07S764D9V` (Panasonic ErgoFit Wired Earbuds)
* **Question**: `"What are the biggest complaints?"`

In [4]:
asin_earbuds = "B07S764D9V"
query_2 = "What are the biggest complaints?"

res_2 = pipeline.answer_question(asin_earbuds, query_2, top_k=4)

display(HTML(f"<b>Product:</b> {res_2['product_title']} (ASIN: <code>{asin_earbuds}</code>)"))
display(Markdown(res_2["answer"]))

Product: Panasonic ErgoFit Wired Earbuds, In-Ear Headphones with Microphone and Call Controller, Ergonomic Custom-Fit Earpieces (S/M/L), 3.5mm Jack for Phones and Laptops - RP-TCM125-A (Blue) (ASIN: B07S764D9V)

Based on 4 relevant reviews for "Panasonic ErgoFit Wired Earbuds, In-Ear Headphones with Microphone and Call Controller, Ergonomic Custom-Fit Earpieces (S/M/L), 3.5mm Jack for Phones and Laptops - RP-TCM125-A (Blue)" (ASIN: B07S764D9V):

### 1. Sentiment Breakdown of Retrieved Reviews
- Negative: 2 (50%) | Neutral: 2 (50%) | Positive: 0 (0%)
- Product Catalog Rating: 4.17 / 5.0 (across 78 total reviews)

### 2. Relevant Aspects Identified
- **Packaging & Box Condition**: cited in 2 review(s) (50.0%)
- **General Product Quality**: cited in 2 review(s) (50.0%)
- **Customer Service & Returns**: cited in 2 review(s) (50.0%)
- **Performance & Functionality**: cited in 2 review(s) (50.0%)
- **Durability & Build Quality**: cited in 2 review(s) (50.0%)

### 3. Main Customer Complaints
1. **Packaging & Box Condition, General Product Quality, Customer Service & Returns, Performance & Functionality, Durability & Build Quality**: RETURNING/DEFECTIVE/MISSING PARTS.
2. **Battery & Charging, Durability & Build Quality**: Died after 2 months.
3. **General Product Quality**: More junk ear buds STOPPED IN 4 months... Same as usual
4. **Packaging & Box Condition, Customer Service & Returns, Battery & Charging, Performance & Functionality**: Sound is great but mic didnt work at all.

### 4. Representative Evidence from Customer Reviews
1. *"There were no adjustable ear pieces in with the packaging. The sound was broken/intermittent. Returning these ASAP!"*  
   — **Rating**: 1.0 / 5.0 | **Sentiment**: negative | **Aspects**: [Packaging & Box Condition, General Product Quality, Customer Service & Returns, Performance & Functionality, Durability & Build Quality]
2. *"Love this earphone but one side stop working after using for 2 months. I bought 3 of them and this is the only one that broke that fast. I guess this is just the unlucky one then."*  
   — **Rating**: 3.0 / 5.0 | **Sentiment**: neutral | **Aspects**: [Battery & Charging, Durability & Build Quality]
3. *"Ok for a few months then as usual the left side STOPPED WORKING!!!! More worthless products (Earbuds)-"*  
   — **Rating**: 1.0 / 5.0 | **Sentiment**: negative | **Aspects**: [General Product Quality]
4. *"I was so happy to get these headphones and today I used it on my first conference but my mic didnt work correctly at all. I should have gotten the regular ones without the mic. But I cant return it now because I threw..."*  
   — **Rating**: 3.0 / 5.0 | **Sentiment**: neutral | **Aspects**: [Packaging & Box Condition, Customer Service & Returns, Battery & Charging, Performance & Functionality]

## 4. Praise Query — Customer Likes & Features
* **Product**: `B075X8471B` (Fire TV Stick)
* **Question**: `"What do customers like about this product?"`

Notice how intent detection automatically shifts to prioritize positive reviews and satisfaction aspects.

In [5]:
query_3 = "What do customers like about this product?"
res_3 = pipeline.answer_question(asin_firetv, query_3, top_k=4)

display(HTML(f"<b>Query Intent Detected:</b> <code>{res_3['retrieval_context']['intent']}</code>"))
display(Markdown(res_3["answer"]))

Query Intent Detected: praises

Based on 4 relevant reviews for "Fire TV Stick with Alexa Voice Remote, streaming media player - Previous Generation" (ASIN: B075X8471B):

### 1. Sentiment Breakdown of Retrieved Reviews
- Negative: 0 (0%) | Neutral: 0 (0%) | Positive: 4 (100%)
- Product Catalog Rating: 4.36 / 5.0 (across 144 total reviews)

### 2. Relevant Aspects Identified
- **General Product Quality**: cited in 1 review(s) (25.0%)

### 3. Key Customer Praises & Liked Features
1. **General Product Quality**: Quality product
2. **General Quality**: ... day two of using it but I’m already so happy I finally decided to buy it
3. **General Quality**: Easy to use
4. **General Quality**: Its great

### 4. Representative Evidence from Customer Reviews
1. *"More to see"*  
   — **Rating**: 5.0 / 5.0 | **Sentiment**: positive | **Aspects**: [General Product Quality]
2. *"Only day two of using it but I’m already so happy I finally decided to buy it!"*  
   — **Rating**: 5.0 / 5.0 | **Sentiment**: positive | **Aspects**: [General]
3. *"I liked it. It's easy to use."*  
   — **Rating**: 4.0 / 5.0 | **Sentiment**: positive | **Aspects**: [General]
4. *"I love this. It does everything I needed it to and more. After reading reviews I did as instructed and couldn't be happier."*  
   — **Rating**: 5.0 / 5.0 | **Sentiment**: positive | **Aspects**: [General]

## 5. Fashion Domain Query — Underwear Sizing & Fabric
* **Product**: `B0956HNH2Y` (Fruit of the Loom Women's Eversoft Cotton Briefs)
* **Question**: `"What is the main issue with this product?"`

Demonstrates domain taxonomy adaptation: detecting **Size & Fit**, **Material & Fabric**, and **Design**.

In [6]:
asin_fashion = "B0956HNH2Y"
query_4 = "What is the main issue with this product?"

res_4 = pipeline.answer_question(asin_fashion, query_4, top_k=4)

display(HTML(f"<b>Domain:</b> {res_4['domain']} | <b>Product:</b> {res_4['product_title']}"))
display(Markdown(res_4["answer"]))

Domain: Fashion | Product: Fruit of the Loom Women's Eversoft Cotton Brief Underwear, Tag Free & Breathable, Available in Plus Size

Based on 4 relevant reviews for "Fruit of the Loom Women's Eversoft Cotton Brief Underwear, Tag Free & Breathable, Available in Plus Size" (ASIN: B0956HNH2Y):

### 1. Sentiment Breakdown of Retrieved Reviews
- Negative: 2 (50%) | Neutral: 1 (25%) | Positive: 1 (25%)
- Product Catalog Rating: 3.8 / 5.0 (across 20 total reviews)

### 2. Relevant Aspects Identified
- **Size & Fit**: cited in 3 review(s) (75.0%)
- **Material & Fabric**: cited in 3 review(s) (75.0%)
- **Design & Style**: cited in 3 review(s) (75.0%)
- **General Product Quality**: cited in 2 review(s) (50.0%)
- **Packaging & Box Condition**: cited in 1 review(s) (25.0%)

### 3. Main Customer Complaints
1. **Packaging & Box Condition, General Product Quality, Size & Fit, Material & Fabric, Design & Style, Color & Appearance, Durability & Build Quality**: What happened?
2. **General Product Quality, Price & Value, Customer Service & Returns, Material & Fabric, Design & Style**: This time I was disappointed
3. **Size & Fit**: Nice product, but I remember when undies were thicker!
4. **Size & Fit, Material & Fabric, Design & Style**: Always discontinue the good style

### 4. Representative Evidence from Customer Reviews
1. *"Ive been wearing these for years and years, and never run into what happened with this package. I ordered 3 sizes smaller than normal due to a big weight loss. Some pairs fit great, others are uncomfortably tight. The..."*  
   — **Rating**: 3.0 / 5.0 | **Sentiment**: neutral | **Aspects**: [Packaging & Box Condition, General Product Quality, Size & Fit, Material & Fabric, Design & Style, Color & Appearance, Durability & Build Quality]
2. *"Seems like Fruit of the Loom makes several types of the same undies with different material for different stores. The ones from Walmart are rough and cheap. The ones from kohls are nicer and softer. These were strange..."*  
   — **Rating**: 1.0 / 5.0 | **Sentiment**: negative | **Aspects**: [General Product Quality, Price & Value, Customer Service & Returns, Material & Fabric, Design & Style]
3. *"I ordered a larger size, thinking they would shrink. So far, there has been no shrinkage. Time will tell, about the shrinkage! Couldn't they make a product, and charge more, that was a little thicker? This is like the..."*  
   — **Rating**: 5.0 / 5.0 | **Sentiment**: positive | **Aspects**: [Size & Fit]
4. *"I already have this style the I purchased at a local "store " about a year ago and they still fit perfectly. That's why when I ordered more I thought NoProblem... WRONG!!! I ordered the exact size I was wearing...but ..."*  
   — **Rating**: 2.0 / 5.0 | **Sentiment**: negative | **Aspects**: [Size & Fit, Material & Fabric, Design & Style]

## 6. Verification of Strict Product Isolation
Verify mathematically that 100% of retrieved reviews belong to the queried product, with zero cross-product contamination.

In [7]:
test_asins = ["B075X8471B", "B010BWYDYA", "B07S764D9V", "B0956HNH2Y", "B07TVHSDMQ"]
verification_results = []

for asin in test_asins:
    res = retriever.retrieve_reviews(asin, "overall quality and issues", top_k=5)
    foreign_reviews = [r for r in res["retrieved_reviews"] if r["parent_asin"] != asin]
    verification_results.append({
        "Product ASIN": asin,
        "Domain": res["domain"],
        "Reviews Retrieved": len(res["retrieved_reviews"]),
        "Foreign Reviews": len(foreign_reviews),
        "Isolation Status": "100% Isolated (PASSED)" if len(foreign_reviews) == 0 else "FAILED",
    })

df_verification = pd.DataFrame(verification_results)
display(HTML("<h3>Cross-Product Leakage Test Results</h3>"))
display(df_verification)

Cross-Product Leakage Test Results

Product ASIN,Domain,Reviews Retrieved,Foreign Reviews,Isolation Status
B075X8471B,Electronics,5,0,100% Isolated (PASSED)
B010BWYDYA,Electronics,5,0,100% Isolated (PASSED)
B07S764D9V,Electronics,5,0,100% Isolated (PASSED)
B0956HNH2Y,Fashion,5,0,100% Isolated (PASSED)
B07TVHSDMQ,Fashion,5,0,100% Isolated (PASSED)


## 7. Modular LLM Layer Demonstration
The pipeline LLM backend is completely decoupled from the retrieval engine via the `BaseLLM` interface.
You can swap between:
- `GroundedSynthesizerLLM`: Zero-hallucination deterministic extractor (Default)
- `HuggingFaceLocalLLM`: Local sequence-to-sequence transformer (`google/flan-t5-small`, etc.)
- `OpenAILLM`: OpenAI GPT models (`gpt-4o-mini`, etc.)
- `GeminiLLM`: Google Gemini models (`gemini-1.5-flash`, etc.)

In [8]:
# Demonstrate modular LLM interface
grounded_llm = get_llm("grounded")
print(f"Current default LLM engine: {grounded_llm.__class__.__name__}")

# Example of instantiating pipeline with custom LLM backend
custom_pipeline = FeedbackRAGPipeline(retriever=retriever, llm=grounded_llm)
print("Pipeline configured with modular LLM successfully.")

Current default LLM engine: GroundedSynthesizerLLM
Pipeline configured with modular LLM successfully.


## 8. Interactive Query Function
Use this helper function to ask any custom question against any product in the catalog.

In [9]:
def ask_feedbackiq(product_id: str, question: str, top_k: int = 5):
    """Query FeedbackIQ RAG for a given product and question."""
    result = pipeline.answer_question(product_id, question, top_k=top_k)
    display(HTML(f"<h3>{result['product_title']} ({result['product_id']})</h3>"))
    display(HTML(f"<b>Question:</b> <i>{result['question']}</i>"))
    display(Markdown(result["answer"]))

# Example interactive invocation
ask_feedbackiq("B010BWYDYA", "Why are customers unhappy with this tablet?")

Fire Tablet with Alexa, 7" Display, 16 GB, Blue - with Special Offers (Previous Generation - 5th) (B010BWYDYA)

Question: Why are customers unhappy with this tablet?

Based on 4 relevant reviews for "Fire Tablet with Alexa, 7" Display, 16 GB, Blue - with Special Offers (Previous Generation - 5th)" (ASIN: B010BWYDYA):

### 1. Sentiment Breakdown of Retrieved Reviews
- Negative: 1 (25%) | Neutral: 3 (75%) | Positive: 0 (0%)
- Product Catalog Rating: 4.26 / 5.0 (across 91 total reviews)

### 2. Relevant Aspects Identified
- **General Product Quality**: cited in 3 review(s) (75.0%)
- **Compatibility & Connection**: cited in 3 review(s) (75.0%)
- **Price & Value**: cited in 2 review(s) (50.0%)
- **Battery & Charging**: cited in 1 review(s) (25.0%)
- **Performance & Functionality**: cited in 1 review(s) (25.0%)

### 3. Main Customer Complaints
1. **General Product Quality**: Ok
2. **General Product Quality, Price & Value, Battery & Charging, Performance & Functionality, Compatibility & Connection**: Poorly quality, dead pixels, wifi issues, poor screen colors and bloatware.
3. **General Product Quality, Price & Value, Compatibility & Connection**: which is horrible (Chrome/Firefox don't render properly and create performance lags)
4. **Compatibility & Connection**: I am happy but...

### 4. Representative Evidence from Customer Reviews
1. *"I left a comment regarding the tablet under the magenta case by mistake lol... my overall feeling is that "it's just ok"... Not great, the touch quality is not great but what can I expect for something so affordable?!!"*  
   — **Rating**: 3.0 / 5.0 | **Sentiment**: neutral | **Aspects**: [General Product Quality]
2. *"Dead pixels on both tablets after 2 months and fading screen colors. It slowed down after 75% internal storage is full and the hi-end fast SDHC card with extra storage for apps installed in SD card due to mediocre har..."*  
   — **Rating**: 1.0 / 5.0 | **Sentiment**: negative | **Aspects**: [General Product Quality, Price & Value, Battery & Charging, Performance & Functionality, Compatibility & Connection]
3. *"Inexpensive, basic tablet. Very low performance processor. Difficult to use any other browser except native Silk app, which is horrible (Chrome/Firefox don't render properly and create performance lags). We use it as ..."*  
   — **Rating**: 3.0 / 5.0 | **Sentiment**: neutral | **Aspects**: [General Product Quality, Price & Value, Compatibility & Connection]
4. *"Not manuel sent with it. Too many commercials. No way to buy video items so have less commercials. Hard to navigate, very touchy. I was told by a friend this is a great item, better than iPad, but I can read my kindle..."*  
   — **Rating**: 3.0 / 5.0 | **Sentiment**: neutral | **Aspects**: [Compatibility & Connection]